# Kaggle Dataset Download & Hector CSV Conversion

## Purpose
This notebook handles:
1. **Kaggle dataset acquisition** - Downloads weather and climate datasets from Kaggle
2. **Hector dataset conversion** - Converts Excel to CSV format (weather/climate data only, excluding rice)

## What This Notebook Does
1. Downloads 2 Kaggle datasets (weather and climate data)
2. Extracts and organizes CSV files
3. Converts Hector Excel dataset to CSV format (wide and long formats) - **Excludes rice-related items**
4. Creates organized directory structure for data storage

## When to Run
- **First time setup**: Run once before the main analysis notebook
- **Data updates**: Re-run to download updated versions
- **Hector conversion**: Run when new Excel file is added

## Requirements
- **Kaggle API credentials**: `kaggle.json` in `~/.kaggle/` directory
  - Download from: https://www.kaggle.com/settings (Account → API → Create New Token)
  - Windows: `C:\Users\<username>\.kaggle\kaggle.json`
  - Linux/Mac: `~/.kaggle/kaggle.json`
- **Hector Excel file**: Place in `data/Hector/Retail Prices 2015-2024.xlsx`

## Output
- `data/kaggle/` - Weather and climate CSV datasets
- `data/Hector/retail_prices_wide.csv` - Hector data in wide format (no rice items)
- `data/Hector/retail_prices_long.csv` - Hector data in long format (no rice items, recommended for ML)

## Hector Dataset Info
- **Coverage**: 2015-2024 (9 years)
- **Items**: Weather and climate related commodities (rice items excluded)
- **Locations**: 37 locations across Sri Lanka
- **Format**: Weekly prices for vegetables, fruits, etc. (excluding rice varieties)

## Next Step
After completion, run: **`Adaptive Crop & Area Optimization.ipynb`**

In [51]:
# =============================================================================
# INSTALL REQUIRED DEPENDENCIES
# =============================================================================

import importlib
import subprocess
import sys

required_packages = ["kaggle", "pandas", "numpy"]
missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if not missing:
    print("All required dependencies are already available.")
else:
    print(f"Installing missing packages: {', '.join(missing)}")

    # Try notebook/kernel environment first.
    install_attempts = [
        [sys.executable, "-m", "pip", "install", "--user", *missing],
        [sys.executable, "-m", "pip", "install", "--break-system-packages", *missing],
    ]

    install_ok = False
    for cmd in install_attempts:
        try:
            subprocess.check_call(cmd)
            install_ok = True
            break
        except subprocess.CalledProcessError:
            continue

    if not install_ok:
        print("Could not install packages automatically in this environment.")
        print("Create/activate a virtual environment, then re-run this notebook.")

# Verify imports after installation attempts
still_missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if still_missing:
    raise RuntimeError(
        "Missing dependencies after install attempts: " + ", ".join(still_missing)
    )

print("\nDependencies are ready!")
print("\nNext: Ensure kaggle.json is in ~/.kaggle/ directory")

All required dependencies are already available.

Dependencies are ready!

Next: Ensure kaggle.json is in ~/.kaggle/ directory


## 2. Kaggle Dataset Download

Downloads weather and climate datasets from Kaggle for Sri Lanka.

**Datasets:**
1. Weather Data
2. Climate Data (314K+ records)

**Vegetable Prices:**
Vegetable price data is available in the local Hector dataset at `data/Hector/Retail Prices 2015-2024.xlsx` (rice items will be excluded during processing).

In [52]:
# =============================================================================
# SETUP DIRECTORIES
# =============================================================================

import os
from pathlib import Path

directories = ["data/kaggle", "../app/models"]

for dir_path in directories:
    os.makedirs(dir_path, exist_ok=True)
    print(f"Created/verified: {dir_path}")

print("\nDirectory structure ready!")

Created/verified: data/kaggle
Created/verified: ../app/models

Directory structure ready!


### 2.2 Kaggle Dataset Configuration

**2 Kaggle Datasets (Sri Lanka Weather & Climate Data):**

| # | Dataset | Description |
|---|---------|-------------|
| 1 | weather_sl | Sri Lanka weather dataset |
| 2 | climate_data_sl | Sri Lanka climate data (314K+ records) |

**Note:** Vegetable price data is available in the local Hector dataset (data/Hector/Retail Prices 2015-2024.xlsx) which covers 2015-2024. Rice-related items will be excluded during processing to focus on weather and climate-relevant commodities only.

In [53]:
# =============================================================================
# KAGGLE DATASET CONFIGURATION
# =============================================================================

KAGGLE_DATASETS = {
    # Climate & Weather ONLY
    "weather_sl":               "rasulmah/sri-lanka-weather-dataset",
    "climate_data_sl":          "tharindumadhusanka9/sri-lanka-climate-data",
}

# Note: All rice-related and economy datasets have been removed
# This notebook now focuses exclusively on weather and climate data
# Hector dataset processing will exclude rice items to maintain weather/climate focus

print(f"Configuration loaded: {len(KAGGLE_DATASETS)} datasets (weather & climate only)")

Configuration loaded: 2 datasets (weather & climate only)


In [54]:
# =============================================================================
# KAGGLE DOWNLOAD FUNCTION
# =============================================================================

import subprocess
import zipfile

def download_kaggle_datasets():
    """
    Download all Kaggle datasets using Kaggle API.
    Skips datasets that already exist.
    """
    print("\n" + "="*70)
    print("DOWNLOADING KAGGLE DATASETS")
    print("="*70)

    KAGGLE_DIR = Path("data/kaggle")

    for nickname, slug in KAGGLE_DATASETS.items():
        target = KAGGLE_DIR / nickname
        target.mkdir(parents=True, exist_ok=True)

        # Skip if already downloaded
        if list(target.glob('*.csv')):
            print(f"[SKIP] {nickname:30s} | Already exists")
            continue

        print(f"[DOWN] {nickname:30s} | Downloading...")

        try:
            subprocess.run(
                ["kaggle", "datasets", "download", "-d", slug, "-p", str(target), "--unzip"],
                check=True,
                capture_output=True
            )
            print(f"[OK]   {nickname:30s} | Downloaded successfully")
        except subprocess.CalledProcessError as e:
            print(f"[ERR]  {nickname:30s} | Error: {e}")
        except FileNotFoundError:
            print(f"[ERR]  Kaggle CLI not found. Install with: pip install kaggle")
            break

    print("="*70)
    print("\nKaggle download complete!")

print("Download function defined")

Download function defined


In [55]:
# =============================================================================
# EXECUTE KAGGLE DOWNLOAD
# =============================================================================

print("="*70)
print(" "*20 + "KAGGLE DOWNLOAD PIPELINE")
print("="*70)

download_kaggle_datasets()

print("\n" + "="*70)
print(" "*25 + "COMPLETE!")
print("="*70)
print("\nAll datasets downloaded!")
print("Location: data/kaggle/")
print("Next: Run 'Adaptive Crop & Area Optimization.ipynb'")

                    KAGGLE DOWNLOAD PIPELINE

DOWNLOADING KAGGLE DATASETS
[SKIP] weather_sl                     | Already exists
[SKIP] climate_data_sl                | Already exists

Kaggle download complete!

                         COMPLETE!

All datasets downloaded!
Location: data/kaggle/
Next: Run 'Adaptive Crop & Area Optimization.ipynb'


## 3. Download Summary

### Downloaded Datasets (2 total)

1. **Weather** - Sri Lanka weather dataset
2. **Climate** - 314K+ climate records

### Local Dataset (Already Available)

**Hector Dataset** - `data/Hector/Retail Prices 2015-2024.xlsx`
- Food commodities (excluding rice items)
- 37 locations across Sri Lanka
- Weekly average retail prices (2015-2024)
- Items include: Tomatoes, Carrot, Beans, Leeks, and other vegetables/fruits (no rice)

### Directory Structure
```
data/
├── kaggle/                    # Downloaded Kaggle datasets
│   ├── weather_sl/
│   └── climate_data_sl/
└── Hector/                    # Local Hector dataset (provided)
    └── Retail Prices 2015-2024.xlsx
```

### Next Steps

1. **Verify downloads**: Run verification cell below
2. **Run main notebook**: Open `Adaptive Crop & Area Optimization.ipynb`
3. **Model training**: Main notebook will load weather, climate, and Hector datasets (excluding rice)

### Troubleshooting

**Kaggle API errors:**
- Check `kaggle.json` location:
  - Windows: `C:\Users\<username>\.kaggle\kaggle.json`
  - Linux/Mac: `~/.kaggle/kaggle.json`
- Verify credentials: Run `kaggle datasets list` in terminal
- Accept dataset terms on Kaggle website

**Re-downloading:**
- Delete specific dataset folder to re-download
- Smart skip prevents duplicate downloads

**Disk space:**
- Expected: ~65-70 MB (2 Kaggle datasets: weather & climate)

**Rice data exclusion:**
- All rice-related items are automatically filtered out during Hector dataset conversion
- This includes varieties like Samba, Nadu, Keeri, Kekulu, Basmati, etc.

In [56]:
# =============================================================================
# VERIFY DOWNLOADS
# =============================================================================

def verify_downloads():
    """Verify all datasets were downloaded."""
    print("="*70)
    print(" "*22 + "VERIFICATION REPORT")
    print("="*70)
    print()

    datasets = list(KAGGLE_DATASETS.keys())
    missing = []
    total_files = 0
    total_size = 0

    for dataset in datasets:
        path = f"data/kaggle/{dataset}"
        if os.path.exists(path) and os.listdir(path):
            file_count = len(os.listdir(path))
            size = sum(os.path.getsize(os.path.join(path, f))
                      for f in os.listdir(path) if os.path.isfile(os.path.join(path, f)))
            size_mb = size / (1024 * 1024)
            print(f"[OK]  {dataset:30s} | {file_count:3d} files | {size_mb:6.2f} MB")
            total_files += file_count
            total_size += size_mb
        else:
            print(f"[MISS] {dataset:30s} | NOT FOUND")
            missing.append(dataset)

    print()
    print("="*70)
    print(f"TOTAL: {total_files} files | {total_size:.2f} MB")
    print("="*70)

    if not missing:
        print("\nAll datasets downloaded successfully!")
        print("Ready for: Adaptive Crop & Area Optimization.ipynb")
    else:
        print(f"\n{len(missing)} dataset(s) missing:")
        for ds in missing:
            print(f"   - {ds}")
        print("\nTip: Re-run download cell above")

verify_downloads()

                      VERIFICATION REPORT

[OK]  weather_sl                     |   2 files |  42.21 MB
[OK]  climate_data_sl                |   1 files |  23.79 MB

TOTAL: 3 files | 66.00 MB

All datasets downloaded successfully!
Ready for: Adaptive Crop & Area Optimization.ipynb


## 4. Hector Dataset CSV Conversion

The Hector dataset is provided as an Excel file. This section converts it to CSV format for easier processing in analysis and ML models.

**Important**: Rice-related items are excluded from the conversion to focus on weather and climate-relevant commodities only.

**Input**: `data/Hector/Retail Prices 2015-2024.xlsx`

**Output**:
- `retail_prices_wide.csv` - Same structure as Excel (excluding rice items)
- `retail_prices_long.csv` - Time series format (excluding rice items, recommended for ML)

**Rice items excluded**: All items containing keywords like "Rice", "Samba", "Nadu", "Keeri", "Kekulu", etc.

In [59]:
# =============================================================================
# CONVERT HECTOR EXCEL TO CSV (EXCLUDING RICE ITEMS)
# =============================================================================

import os
import sys
import pandas as pd

INPUT_FILE = "data/Hector/Retail Prices 2015-2024.xlsx"
OUTPUT_DIR = "data/Hector"
OUTPUT_WIDE = os.path.join(OUTPUT_DIR, "retail_prices_wide.csv")
OUTPUT_LONG = os.path.join(OUTPUT_DIR, "retail_prices_long.csv")

# Rice-related keywords to exclude
RICE_KEYWORDS = [
    'rice', 'samba', 'nadu', 'keeri', 'kekulu', 'rathu', 'kalu',
    'basmati', 'red rice', 'white rice', 'parboiled', 'raw rice'
]

print("="*70)
print("HECTOR DATASET CONVERSION: Excel to CSV (Excluding Rice Items)")
print("="*70)

# Check if Excel file exists
if not os.path.exists(INPUT_FILE):
    print(f"\n[SKIP] Excel file not found: {INPUT_FILE}")
    print("       Place the Excel file in data/Hector/ directory")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Check if CSV files already exist
    if os.path.exists(OUTPUT_WIDE) and os.path.exists(OUTPUT_LONG):
        print(f"\n[SKIP] CSV files already exist")
        print(f"       {OUTPUT_WIDE}")
        print(f"       {OUTPUT_LONG}")
        print("\n       Delete CSV files to regenerate")
    else:
        # Install openpyxl if needed
        try:
            import openpyxl  # noqa: F401
        except ImportError:
            print("\n[1/6] Installing openpyxl for Excel support...")
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
            import openpyxl  # noqa: F401

        # Read Excel file
        print(f"\n[1/6] Reading Excel file: {INPUT_FILE}")
        df = pd.read_excel(INPUT_FILE, sheet_name='2015-2024', header=2)
        print(f"      Loaded {len(df):,} records")

        # Filter out rice items
        print(f"\n[2/6] Filtering out rice-related items...")
        original_count = len(df)

        # Create a mask to exclude rows where 'Items' contains any rice keyword
        rice_mask = df['Items'].astype(str).str.lower().str.contains('|'.join(RICE_KEYWORDS), case=False, na=False)
        removed_items_count = df.loc[rice_mask, 'Items'].nunique()
        df = df[~rice_mask].copy()

        removed_count = original_count - len(df)
        print(f"      Removed {removed_count:,} rice-related records")
        print(f"      Remaining {len(df):,} records (weather/climate relevant)")

        if removed_count > 0:
            print(f"      Unique rice items removed: {removed_items_count}")

        # Save wide format
        print(f"\n[3/6] Saving wide format CSV: {OUTPUT_WIDE}")
        df.to_csv(OUTPUT_WIDE, index=False, encoding='utf-8')
        file_size = os.path.getsize(OUTPUT_WIDE) / (1024 * 1024)
        print(f"      Saved: {file_size:.2f} MB")

        # Convert to long format
        print(f"\n[4/6] Converting to long format...")
        weekly_cols = [col for col in df.columns if str(col).startswith('W') and str(col)[1:].isdigit()]

        df_long = df.melt(
            id_vars=['Year', 'Loc Cod', 'Location', 'Item Cod', 'Items'],
            value_vars=weekly_cols,
            var_name='Week',
            value_name='Price'
        )

        # Extract week number
        df_long['Week_Num'] = df_long['Week'].astype(str).str.extract(r'(\d+)').astype(int)

        # Filter out rows with non-numeric years
        df_long = df_long[pd.to_numeric(df_long['Year'], errors='coerce').notna()].copy()
        df_long['Year'] = df_long['Year'].astype(int)

        # Create date column
        df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-01-01') + pd.to_timedelta((df_long['Week_Num'] - 1) * 7, unit='D')

        # Reorder columns
        df_long = df_long[['Year', 'Week_Num', 'Week', 'Date', 'Loc Cod', 'Location', 'Item Cod', 'Items', 'Price']]

        # Sort and remove missing prices
        df_long = df_long.sort_values(['Year', 'Week_Num', 'Location', 'Items'])
        df_long = df_long.dropna(subset=['Price'])

        # Save long format
        print(f"\n[5/6] Saving long format CSV: {OUTPUT_LONG}")
        df_long.to_csv(OUTPUT_LONG, index=False, encoding='utf-8')
        file_size = os.path.getsize(OUTPUT_LONG) / (1024 * 1024)
        print(f"      Saved: {file_size:.2f} MB")

        # Summary
        print("\n[6/6] Processing complete")
        print("\n" + "="*70)
        print("CONVERSION COMPLETE (Rice Items Excluded)")
        print("="*70)
        print(f"\nWide Format: {len(df):,} records")
        print(f"Long Format: {len(df_long):,} records")
        print(f"Years: {df_long['Year'].min()}-{df_long['Year'].max()}")
        print(f"Items: {df_long['Items'].nunique()} (no rice varieties)")
        print(f"Locations: {df_long['Location'].nunique()}")
        print(f"\nNote: All rice-related items have been excluded")

HECTOR DATASET CONVERSION: Excel to CSV (Excluding Rice Items)

[1/6] Reading Excel file: data/Hector/Retail Prices 2015-2024.xlsx
      Loaded 32,766 records

[2/6] Filtering out rice-related items...
      Removed 2,328 rice-related records
      Remaining 30,438 records (weather/climate relevant)
      Unique rice items removed: 8

[3/6] Saving wide format CSV: data/Hector/retail_prices_wide.csv
      Saved: 8.44 MB

[4/6] Converting to long format...

[5/6] Saving long format CSV: data/Hector/retail_prices_long.csv
      Saved: 60.61 MB

[6/6] Processing complete

CONVERSION COMPLETE (Rice Items Excluded)

Wide Format: 30,438 records
Long Format: 1,017,761 records
Years: 2015-2024
Items: 98 (no rice varieties)
Locations: 37

Note: All rice-related items have been excluded
